# CITS5508 — Lab Sheet 6

**Ensemble methods for regression**

This notebook is split into two parts:

1. **Concrete Slump Test** — Voting regressor combining a linear SVR, a Linear Regressor and an SGD (Ridge) regressor.
2. **Abalone** — Random Forest and Bagging regressors, including feature-importance based dimension reduction.

All work uses `RANDOM_STATE = 42` for reproducibility.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression, SGDRegressor
from sklearn.svm import LinearSVR
from sklearn.ensemble import VotingRegressor, RandomForestRegressor, BaggingRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 100

---
# Part 1 — Concrete Slump Test

The dataset has seven input features (cement, slag, fly ash, water, super-plasticiser, coarse and fine aggregate) and three outputs (slump, flow, 28-day compressive strength). Our regression target is the **28-day Compressive Strength (MPa)**; the other two outputs must be dropped before model training.

## Task 1 — Inspect and clean the data

In [ ]:
raw = pd.read_csv("slump_test.data")
print("Shape:", raw.shape)
raw.head()

In [ ]:
raw.info()
print("\n=== Missing values ===")
print(raw.isna().sum())
print("\n=== Summary statistics ===")
raw.describe()

In [ ]:
def clean_slump_data(df):
    """Clean the Concrete Slump dataset.

    - drops the row index column ('No')
    - drops the SLUMP and FLOW outputs (we only model 28-day strength)
    - returns (X, y) with feature DataFrame and target Series
    """
    df = df.copy()
    df = df.drop(columns=[c for c in df.columns if c.strip().lower() == "no"])
    target = "Compressive Strength (28-day)(Mpa)"
    drop_outputs = ["SLUMP(cm)", "FLOW(cm)"]
    df = df.drop(columns=drop_outputs)
    y = df[target]
    X = df.drop(columns=[target])
    return X, y

X, y = clean_slump_data(raw)
print("Features:", list(X.columns))
print("Target  :", y.name)
print("Shapes  :", X.shape, y.shape)

### Visualisation

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
for ax, col in zip(axes.ravel(), list(X.columns) + [y.name]):
    series = X[col] if col in X.columns else y
    ax.hist(series, bins=20, color="steelblue", edgecolor="black")
    ax.set_title(col)
    ax.set_xlabel("value")
    ax.set_ylabel("count")
fig.suptitle("Distribution of features and target — Concrete Slump", fontsize=14)
fig.tight_layout()
plt.show()

In [ ]:
corr = pd.concat([X, y], axis=1).corr()
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=ax)
ax.set_title("Correlation matrix — features and 28-day strength")
plt.show()

**Observations.** No missing values; all features are numeric and on very different scales (cement and aggregates are in the hundreds, super-plasticiser is single-digit). The target is moderately correlated with cement, water and fly-ash, and the inputs are mildly correlated among themselves but not redundantly so. No additional features need to be removed; we will instead **standardise** the inputs inside each pipeline so the linear/SVR/SGD models are on a level playing field.

## Task 2 — 80/20 split and Voting regressor

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)
print("Train:", X_train.shape, " Test:", X_test.shape)

In [ ]:
def make_pipeline(estimator):
    return Pipeline([
        ("scaler", StandardScaler()),
        ("reg", estimator),
    ])

svr_pipe  = make_pipeline(LinearSVR(random_state=RANDOM_STATE, max_iter=10000))
lin_pipe  = make_pipeline(LinearRegression())
sgd_pipe  = make_pipeline(SGDRegressor(penalty="l2", random_state=RANDOM_STATE, max_iter=10000))

voting_reg = VotingRegressor([
    ("svr", svr_pipe),
    ("lin", lin_pipe),
    ("sgd", sgd_pipe),
])

## Task 3 — Train and compare

In [ ]:
def evaluate(name, model, X_tr, y_tr, X_te, y_te):
    model.fit(X_tr, y_tr)
    preds_tr = model.predict(X_tr)
    preds_te = model.predict(X_te)
    return {
        "model":      name,
        "train_RMSE": float(np.sqrt(mean_squared_error(y_tr, preds_tr))),
        "test_RMSE":  float(np.sqrt(mean_squared_error(y_te, preds_te))),
        "train_MAE":  float(mean_absolute_error(y_tr, preds_tr)),
        "test_MAE":   float(mean_absolute_error(y_te, preds_te)),
        "train_R2":   float(r2_score(y_tr, preds_tr)),
        "test_R2":    float(r2_score(y_te, preds_te)),
    }, preds_te

models_default = {
    "LinearSVR":         svr_pipe,
    "LinearRegression":  lin_pipe,
    "SGD (Ridge)":       sgd_pipe,
    "Voting":            voting_reg,
}

rows, preds_default = [], {}
for name, model in models_default.items():
    row, p = evaluate(name, model, X_train, y_train, X_test, y_test)
    rows.append(row)
    preds_default[name] = p

results_default = pd.DataFrame(rows).set_index("model").round(3)
results_default

**Comment.** All four models behave similarly: train and test RMSE are close, indicating little overfitting on this small dataset. The Voting regressor — being the simple average of the three base predictions — typically lands near the best of the bases and rarely beats it on such a small, low-dimensional problem.

## Task 4 — Predicted vs ground truth (default hyperparameters)

In [ ]:
def plot_pred_vs_true(preds_dict, y_true, suptitle):
    fig, axes = plt.subplots(1, len(preds_dict), figsize=(4.2 * len(preds_dict), 4.2), sharey=True)
    if len(preds_dict) == 1:
        axes = [axes]
    lo = float(min(y_true.min(), min(p.min() for p in preds_dict.values())))
    hi = float(max(y_true.max(), max(p.max() for p in preds_dict.values())))
    for ax, (name, p) in zip(axes, preds_dict.items()):
        ax.scatter(y_true, p, alpha=0.7, edgecolor="black")
        ax.plot([lo, hi], [lo, hi], "r--", lw=1, label="y = x")
        ax.set_title(name)
        ax.set_xlabel("True 28-day strength (MPa)")
        ax.set_ylabel("Predicted (MPa)")
        ax.legend()
    fig.suptitle(suptitle, fontsize=14)
    fig.tight_layout()
    plt.show()

plot_pred_vs_true(preds_default, y_test, "Predicted vs True — default hyperparameters")

In [ ]:
residuals_default = pd.DataFrame({k: v - y_test.values for k, v in preds_default.items()},
                                 index=y_test.index)
print("Residual correlation across models:")
residuals_default.corr().round(3)

**Consistency of mistakes.** The residual correlation matrix shows the three base estimators make highly correlated errors — they over- and under-predict the same instances. This is expected because they are all linear models fed the same standardised features, so they learn essentially the same hyperplane. The voting regressor inherits the same systematic mistakes; ensembling correlated learners cannot diversify away their shared bias.

## Task 5 — Tune SVR and SGD (3-fold GridSearchCV)

In [ ]:
svr_grid = GridSearchCV(
    estimator=make_pipeline(LinearSVR(random_state=RANDOM_STATE, max_iter=20000)),
    param_grid={
        "reg__C":       [0.01, 0.1, 1.0, 10.0, 100.0],
        "reg__epsilon": [0.0, 0.1, 0.5, 1.0],
    },
    cv=3, scoring="neg_root_mean_squared_error", n_jobs=-1,
)
svr_grid.fit(X_train, y_train)
print("Best SVR params:", svr_grid.best_params_)
print(f"Best SVR CV RMSE: {-svr_grid.best_score_:.3f}")

In [ ]:
sgd_grid = GridSearchCV(
    estimator=make_pipeline(SGDRegressor(penalty="l2", random_state=RANDOM_STATE, max_iter=20000)),
    param_grid={
        "reg__alpha":         [1e-5, 1e-4, 1e-3, 1e-2, 1e-1],
        "reg__learning_rate": ["constant", "invscaling", "adaptive"],
        "reg__eta0":          [0.001, 0.01, 0.1],
    },
    cv=3, scoring="neg_root_mean_squared_error", n_jobs=-1,
)
sgd_grid.fit(X_train, y_train)
print("Best SGD params:", sgd_grid.best_params_)
print(f"Best SGD CV RMSE: {-sgd_grid.best_score_:.3f}")

In [ ]:
svr_tuned = svr_grid.best_estimator_
sgd_tuned = sgd_grid.best_estimator_
lin_tuned = lin_pipe
voting_tuned = VotingRegressor([
    ("svr", svr_tuned),
    ("lin", lin_tuned),
    ("sgd", sgd_tuned),
])

models_tuned = {
    "LinearSVR (tuned)":     svr_tuned,
    "LinearRegression":      lin_tuned,
    "SGD (Ridge, tuned)":    sgd_tuned,
    "Voting (tuned)":        voting_tuned,
}

rows, preds_tuned = [], {}
for name, model in models_tuned.items():
    row, p = evaluate(name, model, X_train, y_train, X_test, y_test)
    rows.append(row)
    preds_tuned[name] = p

results_tuned = pd.DataFrame(rows).set_index("model").round(3)
results_tuned

In [ ]:
plot_pred_vs_true(preds_tuned, y_test, "Predicted vs True — tuned hyperparameters")

In [ ]:
compare = pd.concat({"default": results_default, "tuned": results_tuned}, axis=1)
compare

**Comments.** Tuning gives a small but meaningful gain on the SVR and SGD bases — mainly because the defaults pick a learning-rate / regularisation that is not ideal for this scale of features. The linear-regression base is unchanged (no hyperparameters to tune). The Voting regressor now averages slightly better-calibrated bases and so improves modestly. Because the bases remain linear models on the same features, their residuals stay highly correlated — the ensemble cannot eliminate the systematic bias on the highest-strength instances.

---
# Part 2 — Abalone

We now revisit the Abalone dataset and inspect Random Forest and Bagging regressors. The target is the integer **Rings** count; predictions must be rounded to the nearest integer before computing performance metrics.

In [ ]:
ABALONE_COLS = [
    "Sex", "Length", "Diameter", "Height",
    "Whole_weight", "Shucked_weight", "Viscera_weight",
    "Shell_weight", "Rings",
]
abalone = pd.read_csv("abalone.data", header=None, names=ABALONE_COLS)
print("Shape:", abalone.shape)
abalone.head()

In [ ]:
y_ab = abalone["Rings"]
X_ab_raw = abalone.drop(columns=["Rings"])

# One-hot encode Sex; numeric columns pass through.
preprocess = ColumnTransformer([
    ("sex", OneHotEncoder(drop="first"), ["Sex"]),
], remainder="passthrough")

X_ab = preprocess.fit_transform(X_ab_raw)
feature_names = (
    list(preprocess.named_transformers_["sex"].get_feature_names_out(["Sex"]))
    + [c for c in X_ab_raw.columns if c != "Sex"]
)
print("Encoded feature names:", feature_names)

Xab_train, Xab_test, yab_train, yab_test = train_test_split(
    X_ab, y_ab, test_size=0.2, random_state=RANDOM_STATE
)
print("Train:", Xab_train.shape, " Test:", Xab_test.shape)

In [ ]:
def evaluate_int(name, model, X_tr, y_tr, X_te, y_te):
    model.fit(X_tr, y_tr)
    p_tr = np.rint(model.predict(X_tr)).astype(int)
    p_te = np.rint(model.predict(X_te)).astype(int)
    return {
        "model":      name,
        "train_RMSE": float(np.sqrt(mean_squared_error(y_tr, p_tr))),
        "test_RMSE":  float(np.sqrt(mean_squared_error(y_te, p_te))),
        "train_MAE":  float(mean_absolute_error(y_tr, p_tr)),
        "test_MAE":   float(mean_absolute_error(y_te, p_te)),
        "train_R2":   float(r2_score(y_tr, p_tr)),
        "test_R2":    float(r2_score(y_te, p_te)),
    }, p_te

## Task 1 — Random Forest with default hyperparameters

In [ ]:
rf_default = RandomForestRegressor(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1)
row_default, pred_rf_default = evaluate_int("RF (default)", rf_default, Xab_train, yab_train, Xab_test, yab_test)
row_default

## Task 2 — Random Forest with tuned tree hyperparameters

From Lab 5 the best Decision-Tree settings on this dataset were `max_depth=5, min_samples_split=2`. We pass the same tree settings into `RandomForestRegressor`.

In [ ]:
BEST_DT_PARAMS = {"max_depth": 5, "min_samples_split": 2}

rf_tuned = RandomForestRegressor(
    n_estimators=100,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    **BEST_DT_PARAMS,
)
row_tuned, pred_rf_tuned = evaluate_int("RF (DT-tuned)", rf_tuned, Xab_train, yab_train, Xab_test, yab_test)
row_tuned

In [ ]:
pd.DataFrame([row_default, row_tuned]).set_index("model").round(3)

**Impact of tuned hyperparameters.** The fine-tuned values that were optimal for a *single* decision tree restrict each tree in the forest much more than is helpful. A Random Forest already controls overfitting by averaging many bootstrapped, feature-subsampled trees — capping `max_depth=5` cripples each tree's expressive power and the train and test RMSE both rise compared to the unrestricted forest. The tuned-DT settings are not the right choice for a Random Forest base.

## Task 3 — More estimators?

In [ ]:
rmse_curve = []
for n in [25, 50, 100, 200, 400, 800]:
    rf = RandomForestRegressor(n_estimators=n, random_state=RANDOM_STATE, n_jobs=-1)
    rf.fit(Xab_train, yab_train)
    p = np.rint(rf.predict(Xab_test)).astype(int)
    rmse_curve.append((n, float(np.sqrt(mean_squared_error(yab_test, p)))))

ncurve = pd.DataFrame(rmse_curve, columns=["n_estimators", "test_RMSE"])
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(ncurve["n_estimators"], ncurve["test_RMSE"], marker="o")
ax.set_xlabel("n_estimators")
ax.set_ylabel("Test RMSE (rounded)")
ax.set_title("Random Forest — test RMSE vs number of trees")
plt.show()
ncurve

**Comment.** RMSE drops quickly up to roughly 100 trees and then flattens. Adding more estimators reduces variance further but the marginal benefit is tiny — the curve is essentially flat past 200 trees. So increasing the number of estimators offers only diminishing returns; we should not expect a significant performance gain from going much beyond 100.

## Tasks 4–5 — Trim features by importance

In [ ]:
importances = pd.Series(rf_default.feature_importances_, index=feature_names).sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(7, 4))
importances.plot(kind="bar", ax=ax, color="steelblue", edgecolor="black")
ax.axhline(0.05, color="red", linestyle="--", label="0.05 threshold")
ax.set_ylabel("Importance")
ax.set_title("Feature importances — Random Forest (default)")
ax.legend()
plt.tight_layout()
plt.show()
importances.round(4)

In [ ]:
THRESHOLD = 0.05
kept_mask = importances >= THRESHOLD
kept = importances[kept_mask].index.tolist()
removed = importances[~kept_mask].index.tolist()

print(f"Threshold       : {THRESHOLD}")
print(f"Kept ({len(kept)})    :", kept)
print(f"Removed ({len(removed)}):", removed)
print(f"Total importance retained: {importances[kept_mask].sum():.4f}")

## Task 6 — Re-train on the reduced feature set

In [ ]:
kept_idx = [feature_names.index(c) for c in kept]
Xab_train_red = Xab_train[:, kept_idx]
Xab_test_red  = Xab_test[:, kept_idx]
print("Reduced shapes:", Xab_train_red.shape, Xab_test_red.shape)

rf_reduced = RandomForestRegressor(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1)
row_reduced, pred_rf_reduced = evaluate_int(
    "RF (reduced)", rf_reduced, Xab_train_red, yab_train, Xab_test_red, yab_test
)
row_reduced

## Task 7 — Comparison of train and test performance

In [ ]:
rf_results = pd.DataFrame([row_default, row_tuned, row_reduced]).set_index("model").round(3)
rf_results

**Comment.** All three Random-Forest variants overfit (train RMSE noticeably below test RMSE) — Decision-Tree forests fit training data tightly even with bagging. The reduced-feature forest is essentially as good as the full-feature forest, confirming that the dropped features carried very little signal. The DT-tuned forest is the worst because the depth cap is too aggressive.

## Task 8 — Average prediction error per ring value

We use the best Random Forest above (the default 100-tree forest) on the full feature set.

In [ ]:
abs_err = np.abs(pred_rf_default - yab_test.values)
err_df = pd.DataFrame({"true": yab_test.values, "abs_err": abs_err})
agg = err_df.groupby("true")["abs_err"].agg(["mean", "count"]).reset_index()

train_counts = pd.Series(yab_train.values).value_counts().sort_index()

fig, ax1 = plt.subplots(figsize=(10, 5))
ax1.bar(agg["true"], agg["mean"], color="steelblue", edgecolor="black", label="mean |error|")
ax1.set_xlabel("True ring value")
ax1.set_ylabel("Mean absolute prediction error", color="steelblue")
ax1.tick_params(axis="y", labelcolor="steelblue")
ax2 = ax1.twinx()
ax2.plot(train_counts.index, train_counts.values, color="darkorange", marker="o", label="# train instances")
ax2.set_ylabel("Number of training instances", color="darkorange")
ax2.tick_params(axis="y", labelcolor="darkorange")
ax1.set_title("Mean |error| per true ring value vs training-set frequency")
fig.tight_layout()
plt.show()
agg

**Where do errors concentrate?** The largest mean errors appear at both **low** ring values (1–4) and especially the **high** end (≥ 16). The orange line shows training-set frequency: the high-error ring values correspond exactly to the rings with very few training samples. The forest tends to regress towards the most populated band (about 8–12 rings), so under-represented extremes get systematically pulled towards the centre. **Yes** — the large prediction errors are clearly related to insufficient training instances at those ring counts.

## Task 9 — Bagging regressor

We deliberately differentiate the Bagging regressor from the Random Forest:
- `bootstrap=True` and `max_samples=1.0` to build each tree on a bootstrap sample of the same size as the training set;
- `max_features=1.0` so each tree sees **all** features at every split — Random Forest's signature feature subsampling is what we are turning off here.

We try both with default trees and with the Lab 5 fine-tuned tree settings.

In [ ]:
bag_default = BaggingRegressor(
    estimator=DecisionTreeRegressor(random_state=RANDOM_STATE),
    n_estimators=100,
    max_samples=1.0,
    max_features=1.0,
    bootstrap=True,
    bootstrap_features=False,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

bag_tuned = BaggingRegressor(
    estimator=DecisionTreeRegressor(random_state=RANDOM_STATE, **BEST_DT_PARAMS),
    n_estimators=100,
    max_samples=1.0,
    max_features=1.0,
    bootstrap=True,
    bootstrap_features=False,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

## Task 10 — Train Bagging on full-dimensional data and visualise

In [ ]:
row_bag_default, pred_bag_default = evaluate_int("Bagging (default DT)", bag_default,
                                                  Xab_train, yab_train, Xab_test, yab_test)
row_bag_tuned,   pred_bag_tuned   = evaluate_int("Bagging (DT-tuned)",  bag_tuned,
                                                  Xab_train, yab_train, Xab_test, yab_test)
bag_results = pd.DataFrame([row_bag_default, row_bag_tuned]).set_index("model").round(3)
bag_results

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 5), sharey=True)
for ax, (name, p) in zip(axes, [("Bagging (default DT)", pred_bag_default),
                                  ("Bagging (DT-tuned)",  pred_bag_tuned)]):
    ax.scatter(yab_test, p, alpha=0.3, edgecolor="black")
    lo, hi = yab_test.min(), yab_test.max()
    ax.plot([lo, hi], [lo, hi], "r--", lw=1, label="y = x")
    ax.set_title(name)
    ax.set_xlabel("True ring value")
    ax.set_ylabel("Predicted (rounded)")
    ax.legend()
fig.suptitle("Bagging regressor — predicted vs true ring values", fontsize=14)
fig.tight_layout()
plt.show()

## Task 11 — Random Forest vs Bagging

In [ ]:
pd.concat([rf_results, bag_results]).round(3)

**Random Forest vs Bagging.**
- The two **default** ensembles (RF default and Bagging with default trees) achieve very similar test RMSE. Random Forest tends to be marginally better because random feature subsampling at each split decorrelates the trees more aggressively, reducing variance.
- The two ensembles built from `max_depth=5` trees are both worse: depth-5 trees underfit the data, and bagging cannot rescue them.
- Train RMSE tells the same story: unrestricted trees overfit (low train RMSE, higher test RMSE), depth-5 trees do not (train and test RMSE are close but both higher).

**Take-away.** For Abalone the dominant factor is per-tree capacity rather than the choice between RF and plain bagging. Random Forest gives a small edge through feature subsampling but the magnitudes are close. The Lab-5 single-tree hyperparameters are not appropriate for an ensemble of trees.